# Milestone 2: CNN Feature Extraction & SVM Classification
**System**: Skin Cancer Detection and Classification System Using CNN-SVM  
**Objective**: Extract deep feature representations using a MobileNetV2 CNN backbone and train a tuned Support Vector Machine (SVM) classifier.

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from src.feature_extractor import build_cnn_classifier, load_feature_extractor
from src.preprocessing import CLASS_NAMES, IDX_TO_NAME

sns.set_theme(style='whitegrid')


## 1. Load Preprocessed Dataset

In [ ]:
data_path = 'dataset/processed/preprocessed_data.npz'
data = np.load(data_path)
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

print(f'Train shape: {X_train.shape}, Labels: {y_train.shape}')
print(f'Val shape:   {X_val.shape}, Labels: {y_val.shape}')
print(f'Test shape:  {X_test.shape}, Labels: {y_test.shape}')

## 2. Inspect CNN Feature Extractor Architecture

In [ ]:
extractor = load_feature_extractor('models/cnn_feature_extractor.keras')
extractor.summary()

## 3. SVM Hyperparameter Tuning Results

In [ ]:
tuning_file = 'reports/svm_tuning_results.json'
if os.path.exists(tuning_file):
    with open(tuning_file) as f:
        tuning_data = json.load(f)
    tuning_df = pd.DataFrame(tuning_data)
    display(tuning_df)
    
    plt.figure(figsize=(8, 4))
    sns.lineplot(data=tuning_df, x='C', y='val_balanced_accuracy', marker='o', lw=2, color='crimson')
    plt.title('SVM Hyperparameter Tuning (C parameter vs Validation Balanced Accuracy)', fontweight='bold')
    plt.xscale('log')
    plt.xlabel('Regularization Parameter C')
    plt.ylabel('Validation Balanced Accuracy')
    plt.show()

## 4. Test Single Image Inference Pipeline

In [ ]:
from src.predict import predict_skin_lesion
sample_img = X_test[0]
res = predict_skin_lesion(sample_img)
print(f"Predicted Diagnosis: {res['predicted_name']} ({res['predicted_code']})")
print(f"Confidence:          {res['confidence']}%")
print(f"Severity Category:   {res['severity']}")
print(f"Risk Recommendation: {res['risk_level']}")

plt.figure(figsize=(4, 4))
plt.imshow(res['preprocessed_image'])
plt.title(f"{res['predicted_name']} ({res['confidence']}%)", fontweight='bold')
plt.axis('off')
plt.show()